# MCTS Rollout — Caro 11×11

Notebook thí nghiệm cho tác tử MCTS (tìm kiếm cây Monte Carlo kết hợp rollout ngẫu nhiên).
Mọi thuật toán được import từ `agents/mcts_agent.py`; notebook chỉ tổ chức thí nghiệm và trình bày kết quả.

**Trạng thái hiện tại — Giai đoạn 2:** đã có tầng trạng thái (`MCTSState`), nút cây (`MCTSNode`), quy ước giá trị,
bốn bước MCTS (`select_child`/`uct_score`, `MCTSNode.expand`, `rollout`, `backpropagate`), quyết định cuối
(`best_action`) và tác tử `MCTSAgent`. Mục 1–6 đã có số liệu; mục 7–10 (khảo sát ngân sách, thi đấu thử,
phân tích thất bại, tổng kết) thuộc Giai đoạn 3.

Chạy notebook bằng **Restart Kernel & Run All** từ kernel `.venv`.

In [1]:
from pathlib import Path
import os

# Chuyển về thư mục Caro để import module chung từ vị trí notebook.
if Path.cwd().name == 'notebooks' and (Path.cwd().parent / 'core').is_dir():
    os.chdir(Path.cwd().parent)
if not (Path.cwd() / 'core' / 'game_engine.py').is_file():
    raise RuntimeError('Hãy mở notebook từ thư mục caro-11x11 hoặc notebooks')

import platform
import sys

import numpy as np
import pandas as pd

from agents.mcts_agent import (DEFAULT_EXPLORATION, DEFAULT_RADIUS, DEFAULT_TIME_MARGIN_S, MCTSAgent,
                               MCTSNode, MCTSState, outcome_value, uct_score)
from config.search_budget import SearchBudget
from core import CaroGame
from core.game_utils import candidate_actions


def left(df):
    """Căn trái nội dung và tiêu đề bảng để dễ đọc; chỉ dùng để hiển thị."""
    return df.style.set_properties(**{'text-align': 'left'}).set_table_styles(
        [{'selector': 'th', 'props': [('text-align', 'left')]}])

## 1. Câu hỏi nghiên cứu và giới hạn

**Mục tiêu:** xác định câu hỏi mà toàn bộ notebook trả lời và các giới hạn đã biết trước khi đo.

**Câu hỏi nghiên cứu:** MCTS thay đổi chất lượng nước đi và chi phí tính toán như thế nào khi tăng số lượt mô phỏng
(100 / 500 / 1.000 ở chế độ `fixed`) và khi bị giới hạn 2,0 giây/nước (chế độ `time`)?

**Giới hạn đã biết trước khi đo:**
- **Lọc ứng viên:** cây chỉ mở rộng các ô trống trong bán kính 2 quanh quân đã có (`candidate_actions`).
  Đây là phép rút gọn không gian tìm kiếm, không phải tập nước hợp lệ đầy đủ; nước tốt nằm xa hơn sẽ bị bỏ qua.
- **Chính sách rollout:** ngẫu nhiên, không có heuristic chiến thuật. Rollout ngẫu nhiên trên bàn 11×11 thường dài
  và nhiễu, nên MCTS có thể bỏ sót đe dọa ở ngân sách nhỏ. Chính sách cụ thể được chốt và đo ở Giai đoạn 2.
- **Thời gian:** cài đặt bằng Python thuần trên CPU; số mô phỏng đạt được trong 2,0 giây phụ thuộc phần cứng
  và sẽ được ghi nhận thực tế, không giả định trước.

Ô dưới đây đo quy mô của phép lọc ứng viên trên vài thế cờ tạo bằng engine.

**Kỳ vọng trước khi chạy (tính tay):**
- Bàn trống: 121 nước hợp lệ, 1 ứng viên (ô tâm (5, 5) theo quy ước của `candidate_actions`).
- 1 quân ở tâm: 120 nước hợp lệ; ứng viên là ô vuông 5×5 quanh quân trừ ô có quân = 25 − 1 = **24**.
- 1 quân ở góc (0, 0): ô vuông bị cắt ở mép còn 3×3 = 9 ô, trừ ô có quân = **8**.
- 4 quân giữa bàn: các ô vuông 5×5 chồng lên nhau nên ứng viên nhiều hơn 24 nhưng ít hơn 4 × 24.

In [2]:
def play_sequence(moves, size=11, k=5):
    """Tạo thế cờ bằng engine qua chuỗi nước hợp lệ (engine không có API nạp bàn cờ)."""
    game = CaroGame(size, k)
    for move in moves:
        game.step(move)
    return game

positions = {
    'bàn trống': [],
    '1 quân ở tâm': [(5, 5)],
    '4 quân giữa bàn': [(5, 5), (5, 6), (6, 5), (4, 4)],
    '1 quân ở góc': [(0, 0)],
}
rows = []
for name, moves in positions.items():
    game = play_sequence(moves)
    rows.append({'thế cờ': name, 'số quân': game.move_count,
                 'nước hợp lệ': len(game.legal_actions()),
                 f'ứng viên (r={DEFAULT_RADIUS})': len(candidate_actions(game.board, DEFAULT_RADIUS))})
left(pd.DataFrame(rows))

,thế cờ,số quân,nước hợp lệ,ứng viên (r=2)
0,bàn trống,0,121,1
1,1 quân ở tâm,1,120,24
2,4 quân giữa bàn,4,117,40
3,1 quân ở góc,1,120,8


**Quan sát so với kỳ vọng:** số liệu khớp phép tính tay ở cả bốn thế cờ (1 / 24 / 8 / 40).

**Nhận xét:** ở đầu ván, bán kính 2 thu hẹp nhánh từ khoảng 120 nước hợp lệ xuống vài chục ứng viên
(và chỉ 1 ứng viên khi bàn trống). Đây là lý do cây MCTS có thể đi sâu hơn với cùng ngân sách,
nhưng cũng là giới hạn phải công bố khi phân tích các ván thua.

## 2. Môi trường và cấu hình

**Mục tiêu:** ghi toàn bộ cấu hình tại một chỗ để mọi thí nghiệm sau dùng chung và có thể tái lập.

Các tham số dưới đây là cấu hình khởi đầu, chưa phải cấu hình tối ưu thực nghiệm.
Khi một thí nghiệm thay đổi tham số, cấu hình cụ thể sẽ được ghi kèm thí nghiệm đó.

**Định nghĩa thống kê** (chốt trước khi đo, áp dụng từ Giai đoạn 2 trong `agent.stats`, reset mỗi lượt):
| Chỉ số | Cách đếm |
| :--- | :--- |
| `simulations` | Số mô phỏng **hoàn thành**: rollout chạy tới trạng thái kết thúc và đã lan truyền ngược. Mô phỏng bị bỏ dở vì hết thời gian không được đếm và không cập nhật reward. |
| `nodes_visited` | Số nút **mới được tạo** trong cây ở lượt đó (không tính gốc). |
| `pruned_nodes` | Luôn 0: MCTS không cắt tỉa nhánh. Phép lọc ứng viên bán kính 2 không được tính là cắt tỉa. |
| `thinking_time` | Do `BaseAgent.choose_action` đo, đơn vị giây. |

In [3]:
SEED = 42                          # seed cho mọi yếu tố ngẫu nhiên
RADIUS = DEFAULT_RADIUS            # bán kính ứng viên
EXPLORATION = DEFAULT_EXPLORATION  # hệ số khám phá c của UCT (điểm xuất phát, chưa hiệu chỉnh)
ROLLOUT_POLICY = 'ngẫu nhiên đều trên toàn bộ ô trống, hai bên luân phiên, chơi tới khi kết thúc (không cắt ngắn)'
TIME_MARGIN_S = DEFAULT_TIME_MARGIN_S  # chế độ time: dừng ở deadline − 0,1 s (1,9 s trên hạn 2,0 s)
BUDGET_FIXED = SearchBudget(mode='fixed', simulations=1000)
BUDGET_TIME = SearchBudget(mode='time', time_limit_s=2.0)
FIXED_SURVEY = (100, 500, 1000)    # các mức khảo sát ở Giai đoạn 3

def cpu_name():
    """Tên CPU đầy đủ; trên Windows đọc từ registry, nơi khác dùng platform."""
    if sys.platform == 'win32':
        import winreg
        key = winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, r'HARDWARE\DESCRIPTION\System\CentralProcessor\0')
        return winreg.QueryValueEx(key, 'ProcessorNameString')[0].strip()
    return platform.processor() or platform.machine()

try:
    import torch
    torch_info = f'{torch.__version__} (CUDA: {torch.cuda.is_available()})'
except ImportError:
    torch_info = 'chưa cài'

config = {
    'seed': SEED, 'bán kính ứng viên': RADIUS, 'hệ số khám phá c': round(EXPLORATION, 4),
    'chính sách rollout': ROLLOUT_POLICY,
    'biên dừng chế độ time': f'{TIME_MARGIN_S} giây trước deadline',
    'quyết định cuối': 'con có N lớn nhất → Q/N cao hơn → ngẫu nhiên theo seed',
    'ngân sách fixed': f'{BUDGET_FIXED.simulations} mô phỏng/nước',
    'ngân sách time': f'{BUDGET_TIME.time_limit_s} giây/nước',
    'mức khảo sát fixed': FIXED_SURVEY,
    'Python': sys.version.split()[0], 'NumPy': np.__version__, 'pandas': pd.__version__,
    'PyTorch': torch_info, 'hệ điều hành': platform.platform(),
    'CPU': cpu_name(),
}
left(pd.Series(config, name='giá trị').to_frame())

,giá trị
seed,42
bán kính ứng viên,2
hệ số khám phá c,1.414200
chính sách rollout,"ngẫu nhiên đều trên toàn bộ ô trống, hai bên luân phiên, chơi tới khi kết thúc (không cắt ngắn)"
biên dừng chế độ time,0.1 giây trước deadline
quyết định cuối,con có N lớn nhất → Q/N cao hơn → ngẫu nhiên theo seed
ngân sách fixed,1000 mô phỏng/nước
ngân sách time,2.0 giây/nước
mức khảo sát fixed,"(100, 500, 1000)"
Python,3.11.9


**Nhận xét:** MCTS không dùng PyTorch/GPU; thông tin GPU chỉ ghi lại để đối chiếu môi trường chung của nhóm.
Thời gian chạy của MCTS phụ thuộc CPU nên mọi số liệu thời gian sau này phải đi kèm dòng `CPU` ở trên.

## 3. Kiểm tra trạng thái mô phỏng

**Mục tiêu:** minh họa `MCTSState` trước/sau nước đi và đối chiếu với engine `CaroGame`.

Engine không có API nạp bàn cờ tùy ý hay đi lùi, còn agent chỉ nhận `board` và `player`.
Vì vậy MCTS tự dựng trạng thái bằng `MCTSState.from_board(board, player)` (luôn sao chép bàn cờ)
và đi nước bằng luật giống engine. Tính đúng đắn được kiểm tra bằng cách chạy **cùng chuỗi nước hợp lệ**
trên cả hai và so sánh từng bước. Bộ unit test `tests/test_mcts_agent.py` kiểm tra chặt chẽ việc này;
notebook chỉ minh họa.

**Kỳ vọng trước khi chạy:** ở mọi bước, bàn cờ, người đến lượt, số nước, nước cuối và kết quả của hai bên trùng nhau.
Tại trạng thái kết thúc, `current_player` vẫn là người vừa đi nước cuối (đúng quy ước engine).

In [4]:
SYMBOLS = {0: '.', 1: 'X', -1: 'O'}

def board_text(board):
    """Chuỗi hiển thị bàn cờ; chỉ để minh họa, không thuộc thuật toán."""
    size = board.shape[0]
    head = '   ' + ' '.join(f'{c:>2}' for c in range(size))
    body = [f'{r:>2} ' + ' '.join(f'{SYMBOLS[int(v)]:>2}' for v in row) for r, row in enumerate(board)]
    return '\n'.join([head] + body)

def compare(state, game):
    return {
        'bàn cờ khớp': bool(np.array_equal(state.board, game.board)),
        'lượt khớp': state.current_player == game.current_player,
        'số nước khớp': state.move_count == game.move_count,
        'nước cuối khớp': state.last_action == game.last_action,
        'kết quả khớp': state.result() == game.result(),
    }

# Ván ngắn: X thắng hàng ngang ở nước thứ 9.
moves = [(5, 3), (6, 3), (5, 4), (6, 4), (5, 5), (6, 5), (5, 6), (4, 4), (5, 7)]
game = CaroGame()
state = MCTSState.from_board(game.board, game.current_player)
rows = []
for ply, move in enumerate(moves, start=1):
    game.step(move)
    state.step(move)
    rows.append({'nước': ply, 'tọa độ': move, 'người đi': SYMBOLS[int(state.board[move])],
                 'kết quả (góc X)': 'chưa kết thúc' if state.result() is None else state.result(),
                 'current_player': state.current_player,
                 **compare(state, game)})
trace = pd.DataFrame(rows)
trace

,nước,tọa độ,người đi,kết quả (góc X),current_player,bàn cờ khớp,lượt khớp,số nước khớp,nước cuối khớp,kết quả khớp
0,1,"(5, 3)",X,chưa kết thúc,-1,True,True,True,True,True
1,2,"(6, 3)",O,chưa kết thúc,1,True,True,True,True,True
2,3,"(5, 4)",X,chưa kết thúc,-1,True,True,True,True,True
3,4,"(6, 4)",O,chưa kết thúc,1,True,True,True,True,True
4,5,"(5, 5)",X,chưa kết thúc,-1,True,True,True,True,True
5,6,"(6, 5)",O,chưa kết thúc,1,True,True,True,True,True
6,7,"(5, 6)",X,chưa kết thúc,-1,True,True,True,True,True
7,8,"(4, 4)",O,chưa kết thúc,1,True,True,True,True,True
8,9,"(5, 7)",X,1,1,True,True,True,True,True


In [5]:
check_cols = ['bàn cờ khớp', 'lượt khớp', 'số nước khớp', 'nước cuối khớp', 'kết quả khớp']
print('Mọi bước đều khớp engine:', bool(trace[check_cols].all().all()))
print()
print('Trước nước cuối (đến lượt X):')
before = play_sequence(moves[:-1])
print(board_text(before.board))
print()
print('Sau nước cuối (5, 7) — X nối 5 quân:')
print(board_text(state.board))

Mọi bước đều khớp engine: True

Trước nước cuối (đến lượt X):
    0  1  2  3  4  5  6  7  8  9 10
 0  .  .  .  .  .  .  .  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  O  .  .  .  .  .  .
 5  .  .  .  X  X  X  X  .  .  .  .
 6  .  .  .  O  O  O  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .

Sau nước cuối (5, 7) — X nối 5 quân:
    0  1  2  3  4  5  6  7  8  9 10
 0  .  .  .  .  .  .  .  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  O  .  .  .  .  .  .
 5  .  .  .  X  X  X  X  X  .  .  .
 6  .  .  .  O  O  O  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .


In [6]:
# Dựng trạng thái từ đầu vào kiểu agent (chỉ có board và player) ở giữa ván.
mid = play_sequence(moves[:4])
original = mid.board.copy()
from_input = MCTSState.from_board(mid.board, mid.current_player)
child = from_input.play((5, 5))
pd.DataFrame([
    {'trạng thái': 'dựng từ board', 'lượt': SYMBOLS[from_input.current_player],
     'số nước': from_input.move_count, 'nước cuối': from_input.last_action, 'kết quả': from_input.result()},
    {'trạng thái': 'sau play((5,5))', 'lượt': SYMBOLS[child.current_player],
     'số nước': child.move_count, 'nước cuối': child.last_action, 'kết quả': child.result()},
]).assign(**{'board đầu vào không đổi': bool(np.array_equal(mid.board, original)),
             'trạng thái cha không đổi': from_input.board[5, 5] == 0})

,trạng thái,lượt,số nước,nước cuối,kết quả,board đầu vào không đổi,trạng thái cha không đổi
0,dựng từ board,X,4,None,None,True,True
1,"sau play((5,5))",O,5,"(5, 5)",None,True,True


**Nhận xét:**
- Trạng thái mô phỏng khớp engine ở mọi bước của ván minh họa, kể cả bước kết thúc.
- Khi dựng từ đầu vào của agent, `last_action` là `None` vì agent không được biết nước cuối;
  do đó `from_board` quét toàn bộ bàn cờ một lần để nhận ra chuỗi thắng hoặc bàn đầy có sẵn.
- `play()` trả về trạng thái mới; trạng thái cha và mảng `board` đầu vào không bị thay đổi.
- `from_board` tin vào `player` do bên gọi đưa vào, không suy lượt từ số quân.

## 4. Quan sát cấu trúc cây

**Mục tiêu:** trình bày nút gốc, các nút con, lượt thăm (N), giá trị (Q) và quy ước góc nhìn trên một thế cờ nhỏ.

**Phân định dữ liệu: luật cờ và tìm kiếm**

| Lớp | Trường | Thuộc về | Ý nghĩa |
| :--- | :--- | :--- | :--- |
| `MCTSState` | `board` | Luật cờ | Bản sao bàn cờ `int8` (0 trống, 1 X, -1 O) |
| `MCTSState` | `current_player` | Luật cờ | Người đến lượt; tại terminal là người vừa đi (như engine) |
| `MCTSState` | `last_action`, `move_count` | Luật cờ | Nước cuối (None nếu dựng từ bàn cờ) và số quân đã đi |
| `MCTSState` | `result()` | Luật cờ | None / 1 / -1 / 0 theo góc nhìn X |
| `MCTSNode` | `state` | Liên kết | Trạng thái luật cờ tại nút |
| `MCTSNode` | `parent`, `children` | Tìm kiếm | Liên kết cha/con trong cây |
| `MCTSNode` | `action` | Tìm kiếm | Nước dẫn từ cha tới nút (None ở gốc) |
| `MCTSNode` | `player_just_moved` | Tìm kiếm | Góc nhìn dùng để cộng Q |
| `MCTSNode` | `untried_actions`, `radius` | Tìm kiếm | Ứng viên chưa mở rộng (rỗng nếu terminal) |
| `MCTSNode` | `visits` (N), `value` (Q) | Tìm kiếm | Thống kê, khởi tạo N = 0, Q = 0 |

**Quy ước giá trị (giữ nguyên ở mọi giai đoạn):**
- Kết quả ván `result` theo góc nhìn X như engine: `1` X thắng, `-1` O thắng, `0` hòa.
- Mỗi nút lưu `player_just_moved` — người đã đi nước dẫn tới nút. Ở gốc chưa kết thúc, đó là đối thủ của
  người đến lượt; ở gốc đã kết thúc, đó là `current_player` (engine giữ người vừa đi tại terminal).
- Q của nút được cộng dồn theo góc nhìn `player_just_moved`: thắng `+1`, thua `-1`, hòa `0`
  (`outcome_value(result, player) = result × player`).
- Hệ quả: đứng ở nút cha, người đến lượt chính là `player_just_moved` của các nút con, nên Q/N của mỗi con
  đã là giá trị **theo góc nhìn của chính người đang chọn** — ở tầng của X lẫn tầng của O, không cần đổi dấu.
- Q/N chỉ là **thành phần khai thác**. Khi tìm kiếm (Giai đoạn 2), nhánh được chọn theo điểm
  UCT = Q/N + c·√(ln N_cha / N_con), có thêm **thành phần khám phá** ưu tiên nhánh ít được thăm;
  nước chính thức trả về sau tìm kiếm được chọn theo số lượt thăm N.

**Kỳ vọng trước khi chạy:**
- Bảng ví dụ giá trị: X thắng → X +1, O −1; O thắng → O +1, X −1; hòa → cả hai 0.
- Thế cờ X(5,5), O(5,6), X(4,4), đến lượt O: gốc có `player_just_moved = X`; con tầng 1 là nước của O
  nên `player_just_moved = O`; con tầng 2 là nước của X nên `player_just_moved = X`.
- Mọi nút vừa tạo có N = 0, Q = 0, Q/N = 0 (chưa có tìm kiếm).
- Bàn cờ của gốc và của con tầng 1 không bị thay đổi sau khi mở rộng tầng dưới.

In [7]:
examples = []
for result, label in [(1, 'X thắng'), (-1, 'O thắng'), (0, 'hòa')]:
    examples.append({'kết cục': label, 'result (góc X)': result,
                     'giá trị cho X': outcome_value(result, 1), 'giá trị cho O': outcome_value(result, -1)})
pd.DataFrame(examples)

,kết cục,result (góc X),giá trị cho X,giá trị cho O
0,X thắng,1,1.0,-1.0
1,O thắng,-1,-1.0,1.0
2,hòa,0,0.0,0.0


In [8]:
# Thế cờ nhỏ: X ở (5,5), O ở (5,6), X ở (4,4); đến lượt O. Dùng bán kính 1 để cây dễ quan sát.
small = play_sequence([(5, 5), (5, 6), (4, 4)])
root = MCTSNode.root(small.board, small.current_player, radius=1)
print(board_text(small.board))
print()
print('Người đến lượt ở gốc:', SYMBOLS[root.state.current_player])
print('player_just_moved của gốc:', SYMBOLS[root.player_just_moved])
print('Số ứng viên chưa mở rộng ở gốc:', len(root.untried_actions))

# Mở rộng thủ công 3 nước đầu tiên của gốc và 1 nước ở tầng kế tiếp để quan sát liên kết.
for action in list(root.untried_actions[:3]):
    root.expand(action)
first_child = root.children[next(iter(root.children))]
grandchild = first_child.expand(first_child.untried_actions[0])

def node_row(node, depth):
    return {'độ sâu': depth, 'nước dẫn tới': node.action,
            'player_just_moved': SYMBOLS[node.player_just_moved],
            'đến lượt': SYMBOLS[node.state.current_player] if not node.is_terminal() else '-',
            'N': node.visits, 'Q': node.value, 'Q/N': node.mean_value(),
            'số con': len(node.children), 'chưa mở rộng': len(node.untried_actions),
            'cha': node.parent.action if node.parent and node.parent.parent else ('gốc' if node.parent else '-')}

rows = [node_row(root, 0)]
for child in root.children.values():
    rows.append(node_row(child, 1))
    rows.extend(node_row(g, 2) for g in child.children.values())
pd.DataFrame(rows)

    0  1  2  3  4  5  6  7  8  9 10
 0  .  .  .  .  .  .  .  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  X  .  .  .  .  .  .
 5  .  .  .  .  .  X  O  .  .  .  .
 6  .  .  .  .  .  .  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .

Người đến lượt ở gốc: O
player_just_moved của gốc: X
Số ứng viên chưa mở rộng ở gốc: 14


,độ sâu,nước dẫn tới,player_just_moved,đến lượt,N,Q,Q/N,số con,chưa mở rộng,cha
0,0,None,X,O,0,0.0,0.0,3,11,-
1,1,"(3, 3)",O,X,0,0.0,0.0,1,17,gốc
2,2,"(2, 2)",X,O,0,0.0,0.0,0,22,"(3, 3)"
3,1,"(3, 4)",O,X,0,0.0,0.0,0,16,gốc
4,1,"(3, 5)",O,X,0,0.0,0.0,0,17,gốc


In [9]:
# Tính độc lập: mở rộng con không làm đổi bàn cờ của cha.
print('Bàn cờ gốc vẫn như đầu vào:', bool(np.array_equal(root.state.board, small.board)))
print('Bàn cờ con tầng 1 chưa có nước của tầng 2:',
      int(first_child.state.board[grandchild.action]) == 0)
print('Liên kết cha/con đúng:', grandchild.parent is first_child and first_child.parent is root)

Bàn cờ gốc vẫn như đầu vào: True
Bàn cờ con tầng 1 chưa có nước của tầng 2: True
Liên kết cha/con đúng: True


**Quan sát so với kỳ vọng:** khớp toàn bộ — bảng giá trị đúng như dự kiến, góc nhìn luân phiên X → O → X
theo tầng, mọi nút có N = 0, Q = 0, và ba dòng kiểm tra tính độc lập đều `True`.

**Nhận xét:**
- Gốc có `player_just_moved = X` vì đến lượt O; các con tầng 1 có `player_just_moved = O`,
  các con tầng 2 có `player_just_moved = X` — góc nhìn luân phiên đúng theo tầng.
- Mọi nút mới có N = 0, Q = 0 vì cây ở mục này được mở rộng thủ công, chưa chạy mô phỏng nào.
- Phân bố N/Q thực tế sau tìm kiếm được trình bày ở mục 5.

## 5. Theo dõi bốn bước MCTS

**Mục tiêu:** quan sát trực tiếp từ module bốn bước của mỗi mô phỏng — Lựa chọn, Mở rộng, Mô phỏng, Cập nhật ngược —
và trạng thái cây sau tìm kiếm. Dữ liệu lấy qua tham số `trace` của `MCTSAgent.search`; notebook không cài lại thuật toán.

**Điều kiện dừng của từng bước** (cài trong `agents/mcts_agent.py`):
| Bước | Hàm | Dừng khi |
| :--- | :--- | :--- |
| Lựa chọn | `select_child` / `uct_score` | Gặp nút terminal hoặc nút còn nước chưa mở rộng |
| Mở rộng | `MCTSNode.expand` | Thêm đúng 1 con (nước chưa thử chọn ngẫu nhiên theo seed); bỏ qua nếu nút terminal |
| Mô phỏng | `rollout` | Có chuỗi thắng hoặc bàn đầy; nút terminal lấy thẳng kết quả, không rollout; quá mốc dừng thì bỏ mô phỏng |
| Cập nhật ngược | `backpropagate` | Đã cập nhật tới gốc (N += 1, Q += kết quả × `player_just_moved` ở từng nút) |

Vòng tìm kiếm dừng khi đủ `budget.simulations` mô phỏng hoàn thành (`fixed`) hoặc tới deadline − 0,1 s (`time`).

**Thế cờ quan sát:** X(5,5), O(5,6), X(6,5), O(4,4); đến lượt X.

**Kỳ vọng trước khi chạy (12 mô phỏng đầu):**
- Gốc còn nhiều ứng viên chưa thử nên bước Lựa chọn dừng ngay ở gốc (đường lựa chọn rỗng) và mỗi mô phỏng mở rộng một con mới.
- Mỗi rollout kết thúc bằng X thắng, O thắng hoặc hòa; giá trị cộng vào nút mới (nước của X) = kết quả × (+1).
- Sau 12 mô phỏng: N(gốc) = 12, gốc có 12 con, mỗi con N = 1.

In [10]:
observe = play_sequence([(5, 5), (5, 6), (6, 5), (4, 4)])
print(board_text(observe.board))
RESULT_TEXT = {1: 'X thắng', -1: 'O thắng', 0: 'hòa'}

def trace_table(records):
    rows = []
    for r in records:
        rows.append({'mô phỏng': r['simulation'],
                     'đường lựa chọn': ' → '.join(map(str, r['selection_path'])) or '(dừng ở gốc)',
                     'nước mở rộng': r['expanded'] if r['expanded'] is not None else '(nút terminal)',
                     'số nước rollout': len(r['rollout_moves']),
                     'kết quả': RESULT_TEXT[r['result']],
                     'giá trị cộng cho X': outcome_value(r['result'], 1)})
    return pd.DataFrame(rows)

records = []
small_agent = MCTSAgent(seed=SEED, budget=SearchBudget(simulations=12), exploration=EXPLORATION, radius=RADIUS)
root12, _ = small_agent.search(observe.board, observe.current_player, trace=records.append)
print()
print('N(gốc) =', root12.visits, '| số con =', len(root12.children),
      '| ứng viên chưa mở rộng =', len(root12.untried_actions))
left(trace_table(records))

    0  1  2  3  4  5  6  7  8  9 10
 0  .  .  .  .  .  .  .  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  O  .  .  .  .  .  .
 5  .  .  .  .  .  X  O  .  .  .  .
 6  .  .  .  .  .  X  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .

N(gốc) = 12 | số con = 12 | ứng viên chưa mở rộng = 28


,mô phỏng,đường lựa chọn,nước mở rộng,số nước rollout,kết quả,giá trị cộng cho X
0,1,(dừng ở gốc),"(7, 4)",73,O thắng,-1.000000
1,2,(dừng ở gốc),"(5, 3)",45,O thắng,-1.000000
2,3,(dừng ở gốc),"(3, 4)",58,X thắng,1.000000
3,4,(dừng ở gốc),"(4, 5)",35,O thắng,-1.000000
4,5,(dừng ở gốc),"(3, 3)",76,X thắng,1.000000
5,6,(dừng ở gốc),"(8, 3)",61,O thắng,-1.000000
6,7,(dừng ở gốc),"(3, 7)",73,O thắng,-1.000000
7,8,(dừng ở gốc),"(2, 5)",38,X thắng,1.000000
8,9,(dừng ở gốc),"(6, 8)",63,O thắng,-1.000000
9,10,(dừng ở gốc),"(3, 6)",88,X thắng,1.000000


Tiếp tục với 1.000 mô phỏng (`fixed`) trên cùng thế cờ. Khi mọi con của gốc đã được mở rộng, bước Lựa chọn
đi xuống sâu hơn theo UCT. Bảng dưới thống kê **độ dài đường lựa chọn** (số nút đã đi qua trước khi mở rộng)
và 10 con của gốc có N lớn nhất.

**Kỳ vọng:** `simulations` = 1.000 = N(gốc) = tổng N của các con; độ dài đường lựa chọn tăng dần khi cây đầy dần;
nước được chọn là con có N lớn nhất.

In [11]:
records = []
agent = MCTSAgent(seed=SEED, budget=BUDGET_FIXED, exploration=EXPLORATION, radius=RADIUS)
root, fallback = agent.search(observe.board, observe.current_player, trace=records.append)
depths = pd.Series([len(r['selection_path']) for r in records])
print('simulations =', agent.stats['simulations'], '| nodes_visited =', agent.stats['nodes_visited'],
      '| N(gốc) =', root.visits, '| tổng N các con =', sum(c.visits for c in root.children.values()))
print('Phân bố độ dài đường lựa chọn (độ dài: số mô phỏng):', depths.value_counts().sort_index().to_dict())
print('Độ dài trung bình theo từng khối 200 mô phỏng:',
      [round(float(depths[i:i + 200].mean()), 2) for i in range(0, len(depths), 200)])

top = sorted(root.children.values(), key=lambda c: (-c.visits, -c.mean_value()))[:10]
left(pd.DataFrame([{'nước': c.action, 'N': c.visits, 'Q': c.value, 'Q/N (góc X)': round(c.mean_value(), 3),
                    'UCT tại gốc': round(uct_score(c, root.visits, EXPLORATION), 3)} for c in top]))

simulations = 1000 | nodes_visited = 1000 | N(gốc) = 1000 | tổng N các con = 1000
Phân bố độ dài đường lựa chọn (độ dài: số mô phỏng): {0: 40, 1: 948, 2: 12}
Độ dài trung bình theo từng khối 200 mô phỏng: [0.8, 1.0, 1.0, 1.0, 1.06]


,nước,N,Q,Q/N (góc X),UCT tại gốc
0,"(7, 6)",58,14.000000,0.241000,0.729000
1,"(7, 5)",45,7.000000,0.156000,0.710000
2,"(6, 8)",43,7.000000,0.163000,0.730000
3,"(2, 2)",43,7.000000,0.163000,0.730000
4,"(6, 4)",42,6.000000,0.143000,0.716000
5,"(3, 8)",39,5.000000,0.128000,0.723000
6,"(4, 7)",38,4.000000,0.105000,0.708000
7,"(5, 3)",36,4.000000,0.111000,0.731000
8,"(3, 7)",34,2.000000,0.059000,0.696000
9,"(3, 6)",34,2.000000,0.059000,0.696000


In [12]:
# Quyết định cuối do agent đưa ra trên cùng cấu hình và seed (đối chiếu với bảng trên).
decider = MCTSAgent(seed=SEED, budget=BUDGET_FIXED, exploration=EXPLORATION, radius=RADIUS)
chosen = decider.choose_action(observe.board, observe.current_player)
best = max(decider.last_root.children.values(), key=lambda c: c.visits)
print('Nước agent chọn:', chosen, '| N =', decider.last_root.children[chosen].visits,
      '| N lớn nhất trong các con =', best.visits)
print('Thống kê lượt:', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in decider.stats.items()})

Nước agent chọn: (7, 6) | N = 58 | N lớn nhất trong các con = 58
Thống kê lượt: {'thinking_time': 0.229, 'nodes_visited': 1000, 'simulations': 1000, 'pruned_nodes': 0}


**Khám phá và khai thác:** chạy lại 1.000 mô phỏng với các hệ số c khác nhau, giữ nguyên seed và thế cờ.

**Kỳ vọng:** c = 0 chỉ khai thác nên lượt thăm dồn vào ít nhánh; c lớn rải lượt thăm đều hơn.

In [13]:
rows = []
for c in (0.0, 0.5, EXPLORATION, 2 * EXPLORATION):
    a = MCTSAgent(seed=SEED, budget=BUDGET_FIXED, exploration=c, radius=RADIUS)
    move = a.choose_action(observe.board, observe.current_player)
    visits = sorted((ch.visits for ch in a.last_root.children.values()), reverse=True)
    rows.append({'c': round(c, 3), 'số con của gốc': len(visits),
                 'con có N ≥ 10': sum(v >= 10 for v in visits),
                 'tỷ lệ N của con nhiều nhất': round(visits[0] / a.last_root.visits, 3),
                 'nước chọn': move})
left(pd.DataFrame(rows))

,c,số con của gốc,con có N ≥ 10,tỷ lệ N của con nhiều nhất,nước chọn
0,0.000000,40,7,0.821000,"(6, 6)"
1,0.500000,40,21,0.166000,"(7, 6)"
2,1.414000,40,38,0.058000,"(7, 6)"
3,2.828000,40,40,0.035000,"(5, 4)"


In [14]:
# Tái lập: cùng seed và ngân sách fixed cho cùng nước đi và cùng phân bố lượt thăm.
runs = []
for _ in range(2):
    a = MCTSAgent(seed=SEED, budget=BUDGET_FIXED)
    move = a.choose_action(observe.board, observe.current_player)
    runs.append((move, {k: v.visits for k, v in a.last_root.children.items()}))
print('Hai lần chạy cùng seed giống hệt nhau:', runs[0] == runs[1], '| nước:', runs[0][0])

Hai lần chạy cùng seed giống hệt nhau: True | nước: (7, 6)


**Quan sát so với kỳ vọng** (seed 42, số liệu của lần chạy này):
- 12 mô phỏng đầu đúng như dự kiến: cả 12 dừng lựa chọn ở gốc, mỗi mô phỏng mở rộng một con mới; N(gốc) = 12,
  12 con mỗi con N = 1; giá trị cộng cho X là +1 khi rollout X thắng và −1 khi O thắng. Rollout dài 35–104 nước.
- Với 1.000 mô phỏng: `simulations` = N(gốc) = tổng N các con = 1.000, khớp định nghĩa thống kê ở mục 2.
  `nodes_visited` = 1.000 vì mỗi mô phỏng mở rộng đúng một nút (chưa gặp nút terminal trên cây).
- Nước agent chọn trùng con có N lớn nhất trong bảng; hai lần chạy cùng seed cho cùng nước và cùng phân bố lượt thăm.
- c = 0 dồn 82% lượt thăm vào một nhánh; c = √2 rải lượt thăm (38/40 con có N ≥ 10, con nhiều nhất chỉ chiếm 5,8%).

**Nhận xét:**
- **Cây rất nông:** gốc có 40 ứng viên nên 40 mô phỏng đầu chỉ để mở rộng gốc; trong 1.000 mô phỏng chỉ 12 lần
  bước Lựa chọn đi xuống tới độ sâu 2. Với ngân sách này MCTS gần như chỉ đánh giá **một nước** bằng rollout ngẫu nhiên,
  nên chất lượng phụ thuộc mạnh vào độ nhiễu của rollout.
- Q/N của các con ở gốc chênh nhau nhỏ (khoảng 0,06–0,24) trong khi thành phần khám phá với c = √2 còn lớn (khoảng 0,49–0,64 ở 10 con đầu),
  nên UCT vẫn đang rải lượt thăm; nước được chọn chưa tách biệt rõ so với các nước khác.
- Ảnh hưởng của c tới chất lượng nước đi chưa được đo — chỉ quan sát phân bố lượt thăm; khảo sát thuộc Giai đoạn 3.

## 6. Kiểm tra năng lực quyết định

**Mục tiêu:** đo khả năng chọn đúng nước ở các thế chiến thuật cố định, qua 20 seed (0–19) và ba mức ngân sách
`fixed` 100 / 500 / 1.000 mô phỏng. Đây là phép đo, không phải yêu cầu MCTS ngẫu nhiên luôn đúng;
mọi seed thất bại đều được ghi lại.

| Thế cờ | Mô tả | Nước đúng (tính tay) |
| :--- | :--- | :--- |
| `thang_ngay` | X có 4 quân liền hở hai đầu ở hàng 5, đến lượt X | (5, 1) hoặc (5, 6): thắng ngay |
| `chan_bon` | O có 4 quân liền ở hàng 5, đầu trái bị X chặn; đến lượt X | (5, 5): ô duy nhất chặn được O |
| `thang_truoc_chan` | Cả X (hàng 3) và O (hàng 7) cùng có 4 quân hở; đến lượt X | (3, 0) hoặc (3, 5): thắng ngay thay vì chặn |

**Kỳ vọng trước khi chạy:** `thang_ngay` và `thang_truoc_chan` có nước thắng tức thì luôn cho giá trị +1 nên được chọn
ổn định khi ngân sách đủ lớn. `chan_bon` khó hơn với rollout ngẫu nhiên: sau đa số nước của X, O đi ngẫu nhiên
trong rollout hiếm khi hoàn thành chuỗi 5, nên nước chặn không nổi bật so với các nước khác.

In [15]:
import time

TACTICS = {
    'thang_ngay': ([(5, 2), (0, 0), (5, 3), (0, 2), (5, 4), (0, 4), (5, 5), (0, 6)], {(5, 1), (5, 6)}),
    'chan_bon': ([(5, 0), (5, 1), (0, 0), (5, 2), (0, 2), (5, 3), (10, 10), (5, 4)], {(5, 5)}),
    'thang_truoc_chan': ([(3, 1), (7, 1), (3, 2), (7, 2), (3, 3), (7, 3), (3, 4), (7, 4)], {(3, 0), (3, 5)}),
}
SEEDS = range(20)
for name, (moves, good) in TACTICS.items():
    g = play_sequence(moves)
    print(f'{name} — đến lượt {SYMBOLS[g.current_player]}, nước đúng {sorted(good)}')
    print(board_text(g.board))
    print()

thang_ngay — đến lượt X, nước đúng [(5, 1), (5, 6)]
    0  1  2  3  4  5  6  7  8  9 10
 0  O  .  O  .  O  .  O  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  .  .  .  .  .  .  .
 5  .  .  X  X  X  X  .  .  .  .  .
 6  .  .  .  .  .  .  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .

chan_bon — đến lượt X, nước đúng [(5, 5)]
    0  1  2  3  4  5  6  7  8  9 10
 0  X  .  X  .  .  .  .  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  .  .  .  .  .  .  .
 5  X  O  O  O  O  .  .  .  .  .  .
 6  .  .  .  .  .  .  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  X

thang_truoc_chan — đến lượt X, nước đúng

In [16]:
rows, failures = [], []
for name, (moves, good) in TACTICS.items():
    g = play_sequence(moves)
    for sims in FIXED_SURVEY:
        picks, start = [], time.perf_counter()
        for seed in SEEDS:
            a = MCTSAgent(seed=seed, budget=SearchBudget(simulations=sims), exploration=EXPLORATION, radius=RADIUS)
            picks.append(a.choose_action(g.board, g.current_player))
        elapsed = (time.perf_counter() - start) / len(SEEDS)
        failed = [s for s, p in zip(SEEDS, picks) if p not in good]
        wrong = pd.Series([p for p in picks if p not in good], dtype=object).value_counts().head(3)
        rows.append({'thế cờ': name, 'mô phỏng': sims, 'đúng / 20 seed': f'{20 - len(failed)}/20',
                     'tỷ lệ đúng': round(1 - len(failed) / 20, 2),
                     'nước sai hay gặp': ', '.join(f'{k}×{v}' for k, v in wrong.items()) or '-',
                     'giây/nước (TB)': round(elapsed, 3)})
        failures.append({'thế cờ': name, 'mô phỏng': sims, 'seed thất bại': failed})
tactic_table = pd.DataFrame(rows)
left(tactic_table)

,thế cờ,mô phỏng,đúng / 20 seed,tỷ lệ đúng,nước sai hay gặp,giây/nước (TB)
0,thang_ngay,100,1/20,0.050000,"(4, 7)×3, (0, 5)×1, (1, 6)×1",0.024000
1,thang_ngay,500,19/20,0.950000,"(1, 3)×1",0.103000
2,thang_ngay,1000,20/20,1.000000,-,0.199000
3,chan_bon,100,0/20,0.000000,"(8, 9)×2, (7, 1)×2, (6, 0)×2",0.022000
4,chan_bon,500,1/20,0.050000,"(6, 6)×2, (7, 4)×2, (6, 1)×2",0.120000
5,chan_bon,1000,7/20,0.350000,"(4, 3)×2, (7, 6)×2, (0, 3)×1",0.232000
6,thang_truoc_chan,100,4/20,0.200000,"(4, 1)×2, (8, 6)×1, (2, 2)×1",0.019000
7,thang_truoc_chan,500,20/20,1.000000,-,0.076000
8,thang_truoc_chan,1000,20/20,1.000000,-,0.117000


In [17]:
left(pd.DataFrame(failures))

,thế cờ,mô phỏng,seed thất bại
0,thang_ngay,100,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19]"
1,thang_ngay,500,[13]
2,thang_ngay,1000,[]
3,chan_bon,100,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]"
4,chan_bon,500,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19]"
5,chan_bon,1000,"[1, 3, 4, 5, 8, 9, 10, 11, 12, 14, 17, 18, 19]"
6,thang_truoc_chan,100,"[1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 19]"
7,thang_truoc_chan,500,[]
8,thang_truoc_chan,1000,[]


**Quan sát so với kỳ vọng** (20 seed mỗi cấu hình):
| Thế cờ | 100 | 500 | 1.000 |
| :--- | :---: | :---: | :---: |
| `thang_ngay` | 1/20 | 19/20 | 20/20 |
| `chan_bon` | 0/20 | 1/20 | 7/20 |
| `thang_truoc_chan` | 4/20 | 20/20 | 20/20 |

**Phân tích:**
- **Thắng ngay** được chọn ổn định từ 500 mô phỏng: nút con là terminal nên mỗi lượt thăm luôn cộng +1, Q/N = 1 tuyệt đối.
  Ở 100 mô phỏng gần như luôn sai vì gốc có 59 ứng viên — mỗi con chỉ được thăm khoảng 1–2 lần, và vì X đã có
  4 quân hở, phần lớn nước khác của X cũng thắng trong rollout (O đi ngẫu nhiên hiếm khi chặn), nên nhiều con cùng Q/N = 1.
- **Thắng trước khi chặn** cũng đạt 20/20 từ 500: nước thắng ngay vượt trội cả các nước chặn.
- **Chặn bốn** là điểm yếu rõ nhất: chỉ 7/20 ở 1.000 mô phỏng. Sau nước không chặn của X, O trong rollout đi ngẫu nhiên
  trên ~110 ô trống nên chỉ hoàn thành chuỗi 5 với xác suất nhỏ; vì vậy các nước không chặn vẫn có Q/N gần với nước chặn.
  Đây là hạn chế của rollout ngẫu nhiên đều đã nêu ở mục 1, không phải lỗi góc nhìn (mục 4–5 và bộ test xác nhận góc nhìn đúng).
- Thời gian trung bình ~0,2 giây/nước ở 1.000 mô phỏng trên máy ghi ở mục 2, nên chế độ `time` 2,0 giây chạy được
  nhiều mô phỏng hơn hẳn; khảo sát chính thức của chế độ `time` thuộc mục 7 (Giai đoạn 3).
- Hướng cải thiện để đánh giá ở Giai đoạn 3 (biến thể có tên riêng, không thay bản chuẩn): rollout có heuristic
  (ưu tiên nước thắng/chặn ngay) hoặc tăng ngân sách.

---
*Các mục 7–10 (khảo sát ngân sách `fixed`/`time`, thi đấu thử với `RandomAgent`, phân tích thất bại, tổng kết)
chưa thực hiện — thuộc Giai đoạn 3.*